# 02 — Exploratory Data Analysis

Interactive EDA of the synthetic Oil & Gas equipment sensor dataset using Plotly.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.templates.default = 'plotly_dark'

df = pd.read_csv('../data/raw/oil_gas_equipment_sensor_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22440 entries, 0 to 22439
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Equipment_ID            22440 non-null  str           
 1   Equipment_Type          22440 non-null  str           
 2   Plant_Location          22440 non-null  str           
 3   Timestamp               22440 non-null  datetime64[us]
 4   Operating_Hours         22440 non-null  int64         
 5   Temperature_C           21609 non-null  float64       
 6   Pressure_Bar            21443 non-null  float64       
 7   Vibration_mm_s          21461 non-null  float64       
 8   Flow_Rate_m3_h          21458 non-null  float64       
 9   RPM                     21720 non-null  float64       
 10  Motor_Current_A         21859 non-null  float64       
 11  Oil_Temperature_C       21478 non-null  float64       
 12  Oil_Pressure_Bar        21530 non-null  float64       
 1

## Failure distribution

In [2]:
px.pie(df, names='Failure_Within_30_Days', title='Failure Within 30 Days — Overall Distribution')

## Failure rate by equipment type

In [3]:
by_type = df.groupby('Equipment_Type')['Failure_Within_30_Days'].mean().sort_values(ascending=False).reset_index()
px.bar(by_type, x='Equipment_Type', y='Failure_Within_30_Days', title='Failure Rate by Equipment Type')

## Failure rate by plant location

In [4]:
by_plant = df.groupby('Plant_Location')['Failure_Within_30_Days'].mean().sort_values(ascending=False).reset_index()
px.bar(by_plant, x='Plant_Location', y='Failure_Within_30_Days', title='Failure Rate by Plant Location')

## Sensor readings vs failure outcome

In [5]:
for col in ['Temperature_C', 'Pressure_Bar', 'Vibration_mm_s', 'RPM']:
    fig = px.box(df, x='Failure_Within_30_Days', y=col, color='Failure_Within_30_Days',
                 title=f'{col} vs Failure Within 30 Days')
    fig.show()

## Operating hours & maintenance history vs failure

In [6]:
px.box(df, x='Failure_Within_30_Days', y='Operating_Hours', color='Failure_Within_30_Days', title='Operating Hours vs Failure')

In [7]:
maint_fail = df.groupby('Maintenance_History')['Failure_Within_30_Days'].mean().reset_index()
px.bar(maint_fail, x='Maintenance_History', y='Failure_Within_30_Days', title='Failure Rate by Maintenance History')

## Failure trend over time

In [8]:
trend = df.copy()
trend['Month'] = trend['Timestamp'].dt.to_period('M').astype(str)
trend_agg = trend.groupby('Month')['Failure_Within_30_Days'].mean().reset_index()
px.line(trend_agg, x='Month', y='Failure_Within_30_Days', markers=True, title='Failure Rate Trend Over Time')

## Sensor correlation heatmap

In [9]:
corr_cols = ['Temperature_C', 'Pressure_Bar', 'Vibration_mm_s', 'Flow_Rate_m3_h', 'RPM', 'Motor_Current_A', 'Oil_Temperature_C', 'Power_Consumption_kW']
corr = df[corr_cols].corr()
px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Sensor Correlation Heatmap')

## Equipment health score distribution

In [10]:
px.histogram(df, x='Equipment_Health_Score', nbins=30, title='Equipment Health Score Distribution')